write code for localization metrics.

get counts from localiz head, maybe they are better.

add augmentations.

try to change the model.


In [1]:
from pathlib import Path
import sys

# adjust this depending on where the notebook sits
# example: notebook is in project/notebooks/experiments/
PROJECT_ROOT = Path.cwd().resolve().parents[1]   # go up 2 levels

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import yaml
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from tqdm import tqdm
import numpy as np

from datasets.bcdata import BCDataDataset, collate_heatmap_points
from datasets.transforms import PointsToLocalizationHeatmap, PointsToCountHeatmap


from visualization import overlay_heatmap
from training import train


from models.models import HybridModel
from models.losses import weighted_sigmoid_mse_from_logits, softplus_mse_from_logits, l1_count_from_density_logits

from utils.debug import print_info
from src.evaluation import evaluate_count

import torch
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [2]:
with open("../../config.yaml", "r") as f:
    cfg = yaml.safe_load(f)


data_root = Path(cfg["h200_paths"]["data_root"])
checkpoint_dir = Path(cfg["h200_paths"]["checkpoint_dir"])

print(f"data_root: {data_root}")
print(f"checkpoint_dir: {checkpoint_dir}")

data_root: /raid/datasets/Yeldos/BCData
checkpoint_dir: checkpoints


In [3]:
loc_heatmap_generator = PointsToLocalizationHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)
count_heatmap_generator = PointsToCountHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)

train_dataset = BCDataDataset(root = data_root,
                        split="train",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)


test_dataset = BCDataDataset(root = data_root,
                        split="test",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)

val_dataset = BCDataDataset(root = data_root,
                        split="validation",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)

In [4]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,        # choose based on GPU memory (640×640 images are large)
    shuffle=False,
    num_workers=4,       # use 0 if debugging
    pin_memory=True,     # recommended when using GPU
    drop_last=True,       # optional, useful for BatchNorm
    collate_fn = collate_heatmap_points
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,        # choose based on GPU memory (640×640 images are large)
    shuffle=False,
    num_workers=4,       # use 0 if debugging
    pin_memory=True,     # recommended when using GPU
    drop_last=True,       # optional, useful for BatchNorm
    collate_fn = collate_heatmap_points
)


val_loader = DataLoader(
    val_dataset,
    batch_size=8,        # choose based on GPU memory (640×640 images are large)
    shuffle=False,
    num_workers=4,       # use 0 if debugging
    pin_memory=True,     # recommended when using GPU
    drop_last=True,       # optional, useful for BatchNorm
    collate_fn = collate_heatmap_points
)

In [5]:
model = HybridModel()
device = 'cuda'
model.to(device);
model.load_state_dict(torch.load("../../checkpoints/hybrid_02.pt"));

In [6]:
mae_pos, mae_neg, rmse_pos, rmse_neg, mape_pos, mape_neg = evaluate_count(test_loader, model)

print(f"mae_pos, mae_neg, {mae_pos, mae_neg}")
print(f"rmse_pos, rmse_neg, {rmse_pos, rmse_neg}")
print(f"mape_pos, mape_neg, {mape_pos, mape_neg}")

device: cuda:0
total_sample_count: 400
mean n pos: 54.2025,  mean n neg: 108.46
mae_pos, mae_neg, (8.580051784515382, 32.15495389938354)
rmse_pos, rmse_neg, (np.float64(13.33914504698724), np.float64(41.2087439924606))
mape_pos, mape_neg, (16.021364739619816, 30.111718490719795)


In [7]:
mae_pos, mae_neg, rmse_pos, rmse_neg, mape_pos, mape_neg = evaluate_count(train_loader, model)

print(f"mae_pos, mae_neg, {mae_pos, mae_neg}")
print(f"rmse_pos, rmse_neg, {rmse_pos, rmse_neg}")
print(f"mape_pos, mape_neg, {mape_pos, mape_neg}")

device: cuda:0
total_sample_count: 800
mean n pos: 41.25625,  mean n neg: 75.75125
mae_pos, mae_neg, (3.420380342006683, 5.66249451726675)
rmse_pos, rmse_neg, (np.float64(4.320924282251018), np.float64(8.204576819528201))
mape_pos, mape_neg, (10.149338964372873, 6.67917113751173)


In [8]:
mae_pos, mae_neg, rmse_pos, rmse_neg, mape_pos, mape_neg = evaluate_count(val_loader, model)

print(f"mae_pos, mae_neg, {mae_pos, mae_neg}")
print(f"rmse_pos, rmse_neg, {rmse_pos, rmse_neg}")
print(f"mape_pos, mape_neg, {mape_pos, mape_neg}")

device: cuda:0
total_sample_count: 128
mean n pos: 59.28125,  mean n neg: 106.5078125
mae_pos, mae_neg, (7.60786309838295, 29.009133636951447)
rmse_pos, rmse_neg, (np.float64(10.721125781511706), np.float64(36.3282865262286))
mape_pos, mape_neg, (15.464492421597242, 30.419833958148956)
